# Session 2: from matched pixels to millimetres
**A. Real-world problem.** Session 1 said Z = f B / d. A real camera gives pixels, and the
computer must *find* the matching spot in the second picture by comparing small patches of
random dots (speckle). If it can only find the match to a whole pixel, a surface 0.5 m away is
located to about 6 mm, far too coarse to see a rubber strip stretch. How well can matching be
done, and how does a matching error turn into a depth error?

**Plan.** (1) triangulate a matched pixel pair into a 3D point, (2) turn a disparity error into a
depth error, (3) match a patch by cross-correlation, (4) refine the match to a fraction of a
pixel and measure its bias, (5) design a baseline for a required accuracy.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F = 1000.0               # focal length in pixels
BASE = 0.04              # baseline in metres
CX, CY = 160.0, 80.0     # principal point (image centre) of a 320 x 160 pixel image

## Step 1: triangulation with a calibrated pair
"Calibrated" means f, B and the principal point (CX, CY) are known. A matched pair (uL, vL) in the
left image and uR in the right image (same row, because the cameras are parallel) gives

    d = uL - uR      Z = f B / d      X = (uL - CX) Z / f      Y = (vL - CY) Z / f

Hand check: uL = 200, vL = 100, uR = 120 gives d = 80, Z = 0.5 m, X = 40 * 0.5 / 1000 = 0.02 m,
Y = 20 * 0.5 / 1000 = 0.01 m.

In [ ]:
def project_pair(point, f=F, baseline=BASE):
    """Pixel coordinates (uL, vL, uR) of a 3D point (X, Y, Z) in metres seen by the left (x = 0) and right (x = B) camera."""
    X, Y, Z = point
    return f * X / Z + CX, f * Y / Z + CY, f * (X - baseline) / Z + CX


def triangulate(uL, vL, uR, f=F, baseline=BASE):
    """3D point (X, Y, Z) in metres from a matched pixel pair. Disparity <= 0 gives NaN (no valid depth)."""
    d = np.asarray(uL, float) - np.asarray(uR, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        Z = np.where(d > 0, f * baseline / d, np.nan)
    return (np.asarray(uL, float) - CX) * Z / f, (np.asarray(vL, float) - CY) * Z / f, Z

## Step 2: how a disparity error becomes a depth error
From Z = f B / d, a small error dd in the disparity gives, to first order,

    dZ = (f B / d^2) dd = (Z^2 / (f B)) dd

Depth error grows with the *square* of the distance and shrinks with a longer baseline.
Example: Z = 0.5 m, f B = 40 px*m, dd = 0.05 px gives dZ = 0.25 * 0.05 / 40 = 0.000313 m.

In [ ]:
def depth_error(Z, dd, f=F, baseline=BASE):
    """First-order depth error (m) at depth Z (m) for a disparity error dd (pixels)."""
    return Z ** 2 * dd / (f * baseline)


def baseline_for(Z, dZ_max, dd, f=F):
    """Smallest baseline (m) for which the first-order depth error at depth Z stays within dZ_max (m)."""
    return Z ** 2 * dd / (f * dZ_max)

## Step 3: finding the match by cross-correlation
A row of pixels is a sum of round Gaussian dots at random places (a 1D speckle), 20 to 240 grey
levels. We take a *subset* (a patch of 2h+1 pixels) from the left row and slide it along the right
row. For each slide k we compute the zero-normalised cross-correlation (ZNCC):

    C(k) = sum (f - mean f)(g_k - mean g_k) / sqrt( sum (f - mean f)^2 * sum (g_k - mean g_k)^2 )

C is 1 for a perfect match (also if brightness and contrast differ) and near 0 for a random one.

In [ ]:
def make_speckle_1d(n, seed=0, dot_sigma=1.5, density=0.35, margin=15):
    """Random dot positions and strengths for a speckle that can be drawn at fractional positions."""
    rng = np.random.default_rng(seed)
    count = int(density * (n + 2 * margin) / dot_sigma)
    return rng.uniform(-margin, n + margin, count), rng.uniform(0.5, 1.0, count), dot_sigma


def draw_row(speckle, n, shift=0.0, noise=0.0, seed=1):
    """Grey levels of pixels 0..n-1 when the pattern is moved right by `shift` pixels (any real number)."""
    pos, amp, sig = speckle
    x = np.arange(n, dtype=float)[:, None] - shift
    total = (amp[None, :] * np.exp(-((x - pos[None, :]) ** 2) / (2 * sig * sig))).sum(axis=1)
    row = 20.0 + 220.0 * (1.0 - np.exp(-1.5 * total))
    if noise > 0:
        row = row + np.random.default_rng(seed).normal(0.0, noise, n)
    return row


def zncc(f, g):
    f = f - f.mean()
    g = g - g.mean()
    den = np.sqrt((f * f).sum() * (g * g).sum())
    return float((f * g).sum() / den) if den > 0 else 0.0


def correlation_curve(left, right, i0, h=10, search=8):
    """ZNCC of the left subset centred at pixel i0 against the right row shifted by k = -search..search."""
    f = left[i0 - h:i0 + h + 1]
    ks = np.arange(-search, search + 1)
    c = np.array([zncc(f, right[i0 - h + k:i0 + h + 1 + k]) for k in ks])
    return ks, c

## Step 4: sub-pixel refinement and its bias
The best whole-pixel shift k0 is where C peaks. Near its top C looks like a hill, so fit a parabola
through the three points C(k0 - 1), C(k0), C(k0 + 1); its top is at

    k0 + (C(-) - C(+)) / ( 2 (C(-) - 2 C(0) + C(+)) )

This is cheap and good to a few hundredths of a pixel, but a parabola is not exactly the hill
shape, so the answer is pulled slightly toward whole pixels ("pixel locking"). For dots of size
sigma the hill is close to exp(-D^2 / (4 sigma^2)), which lets us predict the bias.

In [ ]:
def match_subpixel(left, right, i0, h=10, search=8):
    """Shift (pixels, real number) of the subset at i0 between the rows, to a fraction of a pixel."""
    ks, c = correlation_curve(left, right, i0, h, search)
    j = int(np.argmax(c))
    if j == 0 or j == len(c) - 1:
        raise ValueError("best match is at the edge of the search range: widen `search`")
    cm, c0, cp = c[j - 1], c[j], c[j + 1]
    return float(ks[j] + (cm - cp) / (2.0 * (cm - 2.0 * c0 + cp)))


def parabola_estimate(true_shift, sigma):
    """Predicted result of the 3-point parabola fit when the hill is exp(-D^2 / (4 sigma^2)) (true_shift in pixels)."""
    k0 = np.round(true_shift)
    r = lambda k: np.exp(-((k - true_shift) ** 2) / (4.0 * sigma ** 2))
    cm, c0, cp = r(k0 - 1), r(k0), r(k0 + 1)
    return k0 + (cm - cp) / (2.0 * (cm - 2.0 * c0 + cp))


def parabola_bias(true_shift, sigma):
    return parabola_estimate(true_shift, sigma) - true_shift


def measured_bias(shifts, seeds=range(12), n=120, h=10, noise=0.0):
    """Mean error of match_subpixel against the exact shift, over several random patterns (m seeds)."""
    err = np.zeros(len(shifts))
    for seed in seeds:
        sp = make_speckle_1d(n, seed=seed)
        left = draw_row(sp, n, 0.0)
        for i, t in enumerate(shifts):
            right = draw_row(sp, n, t, noise=noise, seed=100 + seed)
            err[i] += match_subpixel(left, right, n // 2, h) - t
    return err / len(seeds)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    uL, vL, uR = project_pair((0.02, 0.01, 0.5))
    X, Y, Z = triangulate(uL, vL, uR)
    print(f"point -> ({uL:.1f}, {vL:.1f}) and uR = {uR:.1f};  triangulated back: X {X:.4f} Y {Y:.4f} Z {Z:.4f} m")
    print(f"depth error at Z = 0.5 m for dd = 0.05 px: {1000 * depth_error(0.5, 0.05):.3f} mm;  whole-pixel (dd = 0.5): {1000 * depth_error(0.5, 0.5):.2f} mm")
    print(f"baseline for 0.2 mm at Z = 0.5 m with dd = 0.05 px: {1000 * baseline_for(0.5, 0.0002, 0.05):.1f} mm")

    sp = make_speckle_1d(120, seed=3)
    left, right = draw_row(sp, 120, 0.0), draw_row(sp, 120, 3.3)
    ks, c = correlation_curve(left, right, 60)
    print(f"true shift 3.30 px: best whole pixel {ks[np.argmax(c)]}, sub-pixel estimate {match_subpixel(left, right, 60):.3f}")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(np.arange(120), left, label="left row")
    ax[0].plot(np.arange(120), right, label="right row (moved 3.3 px)")
    ax[0].axvspan(50, 70, color="gray", alpha=0.2, label="subset (21 px)")
    ax[0].set_xlabel("pixel position")
    ax[0].set_ylabel("grey level (0 to 255)")
    ax[0].set_title("A 1D speckle seen twice")
    ax[0].legend(fontsize=8)
    ax[1].plot(ks, c, "o-", label="ZNCC at whole pixels")
    ax[1].axvline(3.3, color="r", ls="--", label="true shift 3.3 px")
    ax[1].axvline(match_subpixel(left, right, 60), color="g", ls=":", label="sub-pixel estimate")
    ax[1].set_xlabel("trial shift k (pixels)")
    ax[1].set_ylabel("correlation C (no unit)")
    ax[1].set_title("Correlation peaks near the true shift")
    ax[1].legend(fontsize=8)
    for a_ in ax:
        a_.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "correlation_peak.png", dpi=110)

    shifts = np.arange(0.0, 1.01, 0.1)
    mb = measured_bias(shifts)
    model = parabola_bias(shifts, 1.5)
    print("fractional shift :", np.round(shifts, 1))
    print("measured bias    :", np.round(mb, 3))
    print("model bias (1.5) :", np.round(model, 3))
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(shifts, mb, "o", label="measured (12 random patterns)")
    ax2.plot(np.linspace(0, 1, 101), parabola_bias(np.linspace(0, 1, 101), 1.5), "-", label="model, dot size 1.5 px")
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("fractional part of the true shift (pixels)")
    ax2.set_ylabel("error of the estimate (pixels)")
    ax2.set_title("Parabola fit: zero at 0 and 0.5, biased in between")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "subpixel_bias.png", dpi=110)

    Zs = np.linspace(0.3, 1.5, 100)
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    for dd, b in ((0.5, 0.04), (0.05, 0.04), (0.05, 0.08)):
        ax3.plot(Zs, 1000 * depth_error(Zs, dd, baseline=b), label=f"disparity error {dd} px, baseline {1000 * b:.0f} mm")
    ax3.set_yscale("log")
    ax3.set_xlabel("depth Z (m)")
    ax3.set_ylabel("depth error (mm)")
    ax3.set_title("Sub-pixel matching and a longer baseline both cut depth error")
    ax3.grid(alpha=0.3, which="both")
    ax3.legend(fontsize=8)
    fig3.tight_layout()
    fig3.savefig(OUT / "depth_error.png", dpi=110)

## Coding tasks
1. **Easy.** Use `triangulate` on the pair (uL, vL, uR) = (130, 60, 90) and predict the depth by hand first.
2. **Stretch.** Measure the sub-pixel error of `match_subpixel` for subsets of half-size 5, 10 and 20 on the same speckle (use `measured_bias`) and say which is best and why.
3. **Explain.** Explain why the bias graph is exactly zero at a shift of 0.5 px in the model but not exactly in the measured points.

## Assessment
- **A. Conceptual.** Why do we match patches of random dots rather than single pixels, and what goes wrong on a blank patch?
- **B. Numerical.** At Z = 1.2 m with f = 1000 px and B = 0.04 m, what depth error comes from a disparity error of 0.1 px?
- **C. Graph and simulation interpretation.** In the sub-pixel bias graph, at which fractional shifts is the error zero, and why is it largest in between?
- **D. Design and optimisation challenge.** Design: choose a baseline so that the depth error at 0.5 m is at most 0.2 mm when sub-pixel matching gives dd = 0.05 px. What changes if matching is only good to 0.1 px?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    X, Y, Z = L.triangulate(200.0, 100.0, 120.0)
    checks.append((f"worked example: (200, 100) and uR = 120 give X 0.02, Y 0.01, Z 0.5 m (got {float(X):.4f}, {float(Y):.4f}, {float(Z):.4f})",
                   abs(X - 0.02) < 1e-9 and abs(Y - 0.01) < 1e-9 and abs(Z - 0.5) < 1e-9))
    p = (-0.03, 0.02, 0.8)
    back = L.triangulate(*L.project_pair(p))
    checks.append(("round trip: project then triangulate returns the point", np.allclose([float(b) for b in back], p)))
    e = L.depth_error(0.5, 0.05)
    checks.append((f"worked example: depth error at 0.5 m for 0.05 px = 0.25 * 0.05 / 40 = 0.3125 mm (got {1000 * e:.4f})", abs(e - 0.25 * 0.05 / 40.0) < 1e-12))
    d = L.F * L.BASE / 0.5
    exact = L.F * L.BASE / (d - 0.05) - 0.5
    checks.append(("first-order depth error is within 1 percent of the exact change", abs(exact - e) / e < 0.01))
    B = L.baseline_for(0.5, 0.0002, 0.05)
    checks.append((f"hand calculation: baseline for 0.2 mm at 0.5 m with 0.05 px is 62.5 mm (got {1000 * B:.1f})", abs(B - 0.0625) < 1e-9))
    checks.append(("with that baseline the depth error is 0.2 mm", abs(L.depth_error(0.5, 0.05, baseline=B) - 0.0002) < 1e-9))
    checks.append(("doubling the distance quadruples the depth error", abs(L.depth_error(1.0, 0.05) / L.depth_error(0.5, 0.05) - 4) < 1e-9))
    x, y, z = L.triangulate(150.0, 80.0, 150.0)
    checks.append(("edge: zero disparity gives no depth (NaN)", bool(np.isnan(z))))

    sp = L.make_speckle_1d(120, seed=3)
    left = L.draw_row(sp, 120, 0.0)
    checks.append(("an unshifted row matches itself with correlation 1", abs(L.zncc(left[50:71], left[50:71]) - 1.0) < 1e-12))
    flat = np.full(21, 100.0)
    checks.append(("edge: a flat patch has correlation 0, not an error", L.zncc(flat, left[50:71]) == 0.0))
    right = L.draw_row(sp, 120, 3.0)
    checks.append(("a whole-pixel shift of 3 is found within 0.1 px", abs(L.match_subpixel(left, right, 60) - 3.0) < 0.1))
    neg = L.draw_row(sp, 120, -2.4)
    checks.append(("a negative shift of -2.4 px is found within 0.15 px", abs(L.match_subpixel(left, neg, 60) + 2.4) < 0.15))
    bright = 0.5 * L.draw_row(sp, 120, 3.0) + 40.0
    checks.append(("matching ignores brightness and contrast changes", abs(L.match_subpixel(left, bright, 60) - L.match_subpixel(left, right, 60)) < 1e-6))
    try:
        L.match_subpixel(left, L.draw_row(sp, 120, 9.0), 60, 10, 8)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge: a shift beyond the search range is reported, not guessed", raised))

    errs, whole = [], []
    for seed in range(20):
        s = L.make_speckle_1d(120, seed=seed)
        lf = L.draw_row(s, 120, 0.0)
        for t in (0.15, 0.3, 0.45, 0.7):
            rt = L.draw_row(s, 120, t)
            errs.append(L.match_subpixel(lf, rt, 60) - t)
            ks, c = L.correlation_curve(lf, rt, 60)
            whole.append(ks[np.argmax(c)] - t)
    rs, rw = float(np.sqrt(np.mean(np.square(errs)))), float(np.sqrt(np.mean(np.square(whole))))
    checks.append((f"sub-pixel error (RMS {rs:.3f} px) is at least 4 times smaller than whole-pixel error (RMS {rw:.3f} px)", rs * 4 < rw))
    t = np.array([0.0, 0.5, 1.0, 0.3])
    pb = L.parabola_bias(t, 1.5)
    checks.append(("parabola bias model is zero at fractional shifts 0 and 0.5", abs(pb[0]) < 1e-12 and abs(pb[1]) < 1e-12))
    checks.append(("parabola bias model is odd about 0.5: bias(0.3) = -bias(0.7)", abs(pb[3] + L.parabola_bias(0.7, 1.5)) < 1e-12))
    checks.append(("bigger dots give a smaller predicted bias", abs(L.parabola_bias(0.3, 3.0)) < abs(L.parabola_bias(0.3, 1.0))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
